1. Basic RAG
The standard implementation retrieves relevant documents and adds them to the prompt.

The query "Why do we need retrieval for LLMs?" is passed to the chain
The retriever finds semantically similar documents to the query from the vector store
The retrieved documents become the context and the original query becomes the question in the prompt
The LLM generates a response based on the prompt with context and query
The response is extracted as a string and printed

What Makes This "Basic RAG"
This implementation has all essential RAG components:

Document storage (albeit with only three sentences)
Vector embeddings for semantic search
Retrieval of relevant context
Augmentation of the prompt with retrieved context
Generation of a response grounded in that context

However, it's "basic" because it lacks more advanced features like:

Document chunking with overlaps
Metadata filtering
Query transformation or expansion
Reranking of retrieved results
Self-questioning or recursive retrieval

The power of this approach is its simplicity while still enabling the LLM to respond with factual information from the provided documents rather than relying solely on its pretrained knowledge.

In [1]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Initialize components
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_texts(
    ["RAG combines retrieval with generative models.",
     "LLMs can hallucinate without proper context.",
     "Vector databases store embeddings for efficient similarity search."],
    embeddings
)
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(model_name="gpt-3.5-turbo")

# Create prompt template
template = """Answer the question based on the following context:
Context: {context}
Question: {question}
Answer: """
prompt = ChatPromptTemplate.from_template(template)

# Set up the RAG chain
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()} 
    | prompt 
    | llm 
    | StrOutputParser()
)

# Example query
query = "Why do we need retrieval for LLMs?"
response = rag_chain.invoke(query)
print(response)

We need retrieval for LLMs because they can hallucinate without proper context. Retrieval helps provide the necessary context for the language model to generate accurate and relevant responses.


2. Parent Document RAG
Retrieves both the relevant chunk and its parent document for additional context.

In [2]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

# Initialize components
embeddings = OpenAIEmbeddings()
llm = ChatOpenAI(model_name="gpt-3.5-turbo")

# Create sample documents with parent-child relationships
parent_doc = "AI research has made significant progress in developing LLMs, which are used for various applications including text generation, summarization, and translation."
child_docs = [
    Document(
        page_content="LLMs are used for text generation applications.",
        metadata={"parent_id": "doc1", "chunk_id": "chunk1"}
    ),
    Document(
        page_content="LLMs can summarize long documents effectively.",
        metadata={"parent_id": "doc1", "chunk_id": "chunk2"}
    ),
    Document(
        page_content="Translation is another key application of modern language models.",
        metadata={"parent_id": "doc1", "chunk_id": "chunk3"}
    )
]

# Store documents in vector DB
vectorstore = Chroma.from_documents(child_docs, embeddings)
retriever = vectorstore.as_retriever()

# Function to get parent document based on metadata
def get_parent_doc(doc_metadata):
    # In a real implementation, this would query a database
    if doc_metadata["parent_id"] == "doc1":
        return parent_doc
    return "No parent document found"

# Modified retrieval function to include parent document
def retrieve_with_parents(query):
    docs = retriever.get_relevant_documents(query)
    augmented_docs = []
    
    for doc in docs:
        parent_content = get_parent_doc(doc.metadata)
        augmented_content = f"Child chunk: {doc.page_content}\nParent document: {parent_content}"
        augmented_docs.append(augmented_content)
    
    return "\n\n".join(augmented_docs)

# Create the RAG chain
template = """Answer the question based on the following context:
Context: {context}
Question: {question}
Answer: """
prompt = ChatPromptTemplate.from_template(template)

def rag_chain(query):
    context = retrieve_with_parents(query)
    prompt_filled = prompt.format(context=context, question=query)
    response = llm.invoke(prompt_filled)
    return response.content

# Example query
query = "How are LLMs used?"
response = rag_chain(query)
print(response)

C:\Users\anilk\AppData\Local\Temp\ipykernel_9620\2889654539.py:41: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(query)
Number of requested results 4 is greater than number of elements in index 3, updating n_results = 3


LLMs are used for various applications including text generation, summarization, and translation.


3. Hypothetical Document RAG
Uses the query to generate hypothetical perfect documents before retrieval.

In [3]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize components
embeddings = OpenAIEmbeddings()
llm = ChatOpenAI(model_name="gpt-3.5-turbo")

# Sample knowledge base
documents = [
    "Neural networks consist of layers of interconnected nodes.",
    "Transformers use self-attention mechanisms to process sequential data.",
    "Convolutional neural networks excel at image recognition tasks.",
    "LLMs are trained on vast amounts of text data using self-supervised learning."
]
vectorstore = FAISS.from_texts(documents, embeddings)
retriever = vectorstore.as_retriever()

# Hypothetical document generation prompt
hypo_template = """Based on the user question: '{question}'
Generate a hypothetical document that would perfectly answer the user question.
This document should be detailed, accurate, and directly relevant to the query.
Hypothetical Document:"""
hypo_prompt = ChatPromptTemplate.from_template(hypo_template)

# Main RAG prompt
rag_template = """Answer the question based on the following retrieved context:
Context: {context}
Question: {question}
Answer:"""
rag_prompt = ChatPromptTemplate.from_template(rag_template)

# Hypothetical document RAG function
def hypo_doc_rag(query):
    # Step 1: Generate hypothetical document
    hypo_doc_chain = hypo_prompt | llm | StrOutputParser()
    hypothetical_doc = hypo_doc_chain.invoke({"question": query})
    
    # Step 2: Embed the hypothetical document and use it for retrieval
    hypo_embedding = embeddings.embed_query(hypothetical_doc)
    docs = vectorstore.similarity_search_by_vector(hypo_embedding, k=2)
    context = "\n".join([doc.page_content for doc in docs])
    
    # Step 3: Answer using retrieved documents
    rag_chain = rag_prompt | llm | StrOutputParser()
    response = rag_chain.invoke({"context": context, "question": query})
    
    return response

# Example query
query = "How do neural networks process information?"
response = hypo_doc_rag(query)
print(response)

Neural networks process information by passing it through layers of interconnected nodes, where each node applies a mathematical operation to the input data.


4. Multi-Query RAG
Generates multiple queries from the original question to improve retrieval.

In [4]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize components
embeddings = OpenAIEmbeddings()
llm = ChatOpenAI(model_name="gpt-3.5-turbo")

# Sample knowledge base
documents = [
    "The water cycle involves evaporation from oceans and lakes.",
    "Condensation occurs when water vapor cools and forms clouds.",
    "Precipitation happens when water droplets in clouds become too heavy.",
    "Runoff is the flow of water from land to bodies of water.",
    "Groundwater is water that filters down through soil and rock."
]
vectorstore = FAISS.from_texts(documents, embeddings)
retriever = vectorstore.as_retriever()

# Query generation prompt
query_gen_template = """Generate 3 different search queries that would help answer the original question. 
These should be different perspectives or aspects of the original question.

Original question: {question}

Output the 3 queries as a numbered list:
1.
2.
3."""
query_gen_prompt = ChatPromptTemplate.from_template(query_gen_template)

# RAG prompt
rag_template = """Answer the original question based on the following retrieved contexts:
Contexts: {contexts}
Original Question: {question}
Answer:"""
rag_prompt = ChatPromptTemplate.from_template(rag_template)

# Parse generated queries
def parse_queries(text):
    lines = text.strip().split('\n')
    queries = []
    for line in lines:
        if line.startswith('1.') or line.startswith('2.') or line.startswith('3.'):
            queries.append(line[2:].strip())
    return queries

# Multi-query RAG function
def multi_query_rag(question):
    # Step 1: Generate multiple queries
    query_gen_chain = query_gen_prompt | llm | StrOutputParser()
    query_text = query_gen_chain.invoke({"question": question})
    queries = parse_queries(query_text)
    
    # Step 2: Retrieve documents for each query
    all_docs = []
    for query in queries:
        docs = retriever.get_relevant_documents(query)
        all_docs.extend(docs)
    
    # Step 3: Deduplicate and prepare context
    unique_contents = set()
    unique_docs = []
    
    for doc in all_docs:
        if doc.page_content not in unique_contents:
            unique_contents.add(doc.page_content)
            unique_docs.append(doc)
    
    contexts = "\n".join([doc.page_content for doc in unique_docs])
    
    # Step 4: Answer using combined context
    rag_chain = rag_prompt | llm | StrOutputParser()
    response = rag_chain.invoke({"contexts": contexts, "question": question})
    
    return response

# Example query
query = "Explain how water moves through the environment."
response = multi_query_rag(query)
print(response)

Water moves through the environment in several ways. It starts with evaporation from oceans and lakes, where water turns into water vapor and rises into the atmosphere. This water vapor then cools and condenses to form clouds through a process called condensation. When water droplets in clouds become too heavy, precipitation occurs in the form of rain, snow, sleet, or hail. The water then flows over land as runoff, eventually making its way back to bodies of water. Some water also filters down through soil and rock, becoming groundwater. This continuous cycle of evaporation, condensation, precipitation, runoff, and groundwater movement is known as the water cycle.


5. Recursive RAG
Uses multiple retrieval steps to progressively refine the answer.

In [5]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import json

# Initialize components
embeddings = OpenAIEmbeddings()
llm = ChatOpenAI(model_name="gpt-3.5-turbo")

# Sample knowledge base
documents = [
    "Python is a high-level programming language known for its readability.",
    "Python supports multiple programming paradigms including procedural and object-oriented.",
    "Python was created by Guido van Rossum and released in 1991.",
    "Python's name comes from Monty Python, not the snake.",
    "Python uses indentation for code blocks instead of braces or keywords.",
    "Popular Python frameworks include Django for web development and TensorFlow for machine learning."
]
vectorstore = FAISS.from_texts(documents, embeddings)
retriever = vectorstore.as_retriever()

# Initial retrieval and answer prompt
initial_template = """Based on this context, provide an initial answer to the question. 
Also identify specific follow-up questions needed to improve the answer.

Context: {context}
Question: {question}

Output in the following JSON format:
{{"initial_answer": "your initial answer here",
  "follow_up_questions": ["specific question 1", "specific question 2"]}}"""
initial_prompt = ChatPromptTemplate.from_template(initial_template)

# Follow-up prompt
followup_template = """Use the new information to enhance the previous answer.

Original Question: {original_question}
Previous Answer: {previous_answer}
New Information from Follow-up: {follow_up_info}

Provide an enhanced answer:"""
followup_prompt = ChatPromptTemplate.from_template(followup_template)

# Recursive RAG implementation
def recursive_rag(question, max_iterations=2):
    # Step 1: Initial retrieval and answer
    docs = retriever.get_relevant_documents(question)
    context = "\n".join([doc.page_content for doc in docs])
    
    initial_chain = initial_prompt | llm | StrOutputParser()
    initial_response = initial_chain.invoke({"context": context, "question": question})
    
    try:
        parsed_response = json.loads(initial_response)
        current_answer = parsed_response["initial_answer"]
        follow_up_questions = parsed_response["follow_up_questions"]
    except:
        # Fallback if JSON parsing fails
        current_answer = initial_response
        follow_up_questions = []
    
    # Step 2: Recursive follow-up for each question
    iterations = 0
    while follow_up_questions and iterations < max_iterations:
        iterations += 1
        
        # Get the next follow-up question
        follow_up = follow_up_questions.pop(0)
        
        # Retrieve information for the follow-up
        follow_up_docs = retriever.get_relevant_documents(follow_up)
        follow_up_info = "\n".join([doc.page_content for doc in follow_up_docs])
        
        # Enhance the answer with new information
        followup_chain = followup_prompt | llm | StrOutputParser()
        enhanced_answer = followup_chain.invoke({
            "original_question": question,
            "previous_answer": current_answer,
            "follow_up_info": follow_up_info
        })
        
        # Update the current answer
        current_answer = enhanced_answer
    
    return current_answer

# Example query
query = "Tell me about Python programming language."
response = recursive_rag(query)
print(response)

Python is a versatile high-level programming language known for its readability and ease of use. It supports multiple programming paradigms, including procedural and object-oriented programming. Python was created by Guido van Rossum and released in 1991, making it one of the older programming languages still widely used today. Popular Python frameworks like Django for web development and TensorFlow for machine learning have contributed to Python's popularity among developers for a wide range of applications. These frameworks provide powerful tools and libraries that make it easier for developers to build robust web applications and implement complex machine learning algorithms using Python.


6. Reranking RAG
Uses an additional model to rerank retrieved documents based on relevance.

In [6]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

# Initialize components
embeddings = OpenAIEmbeddings()
llm = ChatOpenAI(model_name="gpt-3.5-turbo")
reranker_llm = ChatOpenAI(model_name="gpt-3.5-turbo")

# Sample knowledge base
documents = [
    Document(page_content="Climate change is caused primarily by greenhouse gas emissions."),
    Document(page_content="Carbon dioxide levels have increased significantly since pre-industrial times."),
    Document(page_content="Rising temperatures lead to melting ice caps and sea level rise."),
    Document(page_content="Extreme weather events become more frequent due to climate change."),
    Document(page_content="Renewable energy sources like solar and wind can reduce emissions."),
    Document(page_content="The Paris Agreement aims to limit global warming to below 2 degrees Celsius."),
    Document(page_content="Weather patterns can vary seasonally and aren't always indicative of climate change."),
    Document(page_content="Electric vehicles produce fewer emissions than traditional combustion engines.")
]
vectorstore = FAISS.from_texts([doc.page_content for doc in documents], embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Reranking prompt
rerank_template = """Given a query and a list of retrieved documents, rate each document's relevance to the query on a scale of 1-10.
10 means the document is perfectly relevant, 1 means it's not relevant at all.

Query: {query}

Documents:
{documents}

Output a JSON list of scores like:
[
  {{
    "index": 0,
    "score": 8,
    "reasoning": "Brief explanation"
  }},
  ...
]"""
rerank_prompt = ChatPromptTemplate.from_template(rerank_template)

# RAG prompt
rag_template = """Answer the question based on these relevant documents:
Context: {context}
Question: {question}
Answer:"""
rag_prompt = ChatPromptTemplate.from_template(rag_template)

# Reranking RAG function
def reranking_rag(query):
    # Step 1: Initial retrieval
    initial_docs = retriever.get_relevant_documents(query)
    
    # Step 2: Format documents for reranking
    docs_formatted = ""
    for i, doc in enumerate(initial_docs):
        docs_formatted += f"{i}. {doc.page_content}\n"
    
    # Step 3: Rerank documents
    rerank_chain = rerank_prompt | reranker_llm 
    rerank_response = rerank_chain.invoke({"query": query, "documents": docs_formatted})
    
    try:
        # Parse the output to get the scores - we'll simulate this here
        import json
        import re
        
        # Extract JSON from potential text output
        json_match = re.search(r'\[.*\]', rerank_response.content, re.DOTALL)
        if json_match:
            scores = json.loads(json_match.group())
            
            # Sort documents by score
            scored_docs = []
            for score_info in scores:
                if score_info["index"] < len(initial_docs):
                    doc = initial_docs[score_info["index"]]
                    scored_docs.append((doc, score_info["score"]))
            
            # Sort by score descending
            sorted_docs = [doc for doc, score in sorted(scored_docs, key=lambda x: x[1], reverse=True)]
            
            # Take top 3 docs
            top_docs = sorted_docs[:3]
        else:
            top_docs = initial_docs[:3]  # Fallback to top 3 initial docs
    except:
        # Fallback if parsing fails
        top_docs = initial_docs[:3]
    
    # Step 4: Generate answer with reranked documents
    context = "\n".join([doc.page_content for doc in top_docs])
    rag_chain = rag_prompt | llm | StrOutputParser()
    response = rag_chain.invoke({"context": context, "question": query})
    
    return response

# Example query
query = "How does climate change affect our planet?"
response = reranking_rag(query)
print(response)

Climate change affects our planet by causing extreme weather events to become more frequent, leading to rising temperatures, melting ice caps, and sea level rise. This is primarily caused by greenhouse gas emissions, which contribute to the warming of the Earth's atmosphere. These changes have far-reaching impacts on ecosystems, weather patterns, and sea levels, posing significant challenges for both human societies and the natural world.


7. Self-Query RAG
Converts a natural language query into a structured query with filters.

In [7]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List, Optional

# Define structured query schema
class StructuredQuery(BaseModel):
    query: str = Field(description="The core semantic query for vector search")
    filters: dict = Field(description="Metadata filters to apply", default_factory=dict)

# Initialize components
embeddings = OpenAIEmbeddings()
llm = ChatOpenAI(model_name="gpt-3.5-turbo")

# Sample documents with metadata
documents = [
    ("Apple releases new iPhone with improved camera", {"category": "technology", "date": "2024-03-15", "source": "tech_news"}),
    ("Major hurricane hits Florida coast causing widespread damage", {"category": "weather", "date": "2024-02-10", "source": "weather_channel"}),
    ("Scientists discover new species in Amazon rainforest", {"category": "science", "date": "2024-01-05", "source": "science_daily"}),
    ("Tech company unveils advanced AI assistant", {"category": "technology", "date": "2024-03-01", "source": "tech_news"}),
    ("Global climate report shows accelerating warming trends", {"category": "environment", "date": "2024-02-20", "source": "climate_watch"}),
    ("New healthcare policy implemented nationwide", {"category": "health", "date": "2024-01-15", "source": "health_gazette"})
]

# Create FAISS index with documents and metadata
texts = [doc[0] for doc in documents]
metadatas = [doc[1] for doc in documents]
vectorstore = FAISS.from_texts(texts, embeddings, metadatas=metadatas)

# Query parser prompt
parser_template = """Analyze the user's query and convert it into a structured query with potential metadata filters.

User query: {query}

Return a JSON object with:
1. A semantic query for vector search
2. Any metadata filters (category, date, source) that should be applied

For example, if the user asks "Find technology news from March 2024", the structure would include a semantic query about technology and a filter for date and category.

{format_instructions}"""

parser = PydanticOutputParser(pydantic_object=StructuredQuery)
parser_prompt = ChatPromptTemplate.from_template(
    template=parser_template,
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# RAG prompt
rag_template = """Answer the question based on these relevant documents:
Context: {context}
Question: {question}
Answer:"""
rag_prompt = ChatPromptTemplate.from_template(rag_template)

# Self-query RAG function
def self_query_rag(query):
    # Step 1: Parse natural language query into structured query with filters
    parser_chain = parser_prompt | llm | parser
    structured_query = parser_chain.invoke({"query": query})
    
    # Step 2: Execute the structured query
    filter_dict = structured_query.filters
    
    # Convert date string to date object if needed for comparison
    # (simplified here - in real implementation would need proper date parsing)
    
    # Perform vector search with metadata filters
    results = vectorstore.similarity_search_with_score(
        structured_query.query,
        k=10,  # Get more initially to filter
    )
    
    # Apply filters manually (in a real implementation, the vector DB would do this)
    filtered_results = []
    for doc, score in results:
        matches_filters = True
        for filter_key, filter_value in filter_dict.items():
            if filter_key in doc.metadata:
                # Simplified filter matching - would be more sophisticated in real implementation
                if isinstance(filter_value, str) and filter_value.lower() not in doc.metadata[filter_key].lower():
                    matches_filters = False
                    break
                # Could add date range handling here
        
        if matches_filters:
            filtered_results.append(doc)
    
    # Limit to top results
    top_docs = filtered_results[:3]
    
    # Step 3: Generate answer with filtered documents
    if top_docs:
        context = "\n".join([doc.page_content for doc in top_docs])
    else:
        context = "No relevant documents found matching all criteria."
    
    rag_chain = rag_prompt | llm | StrOutputParser()
    response = rag_chain.invoke({"context": context, "question": query})
    
    return response

# Example query with implicit filters
query = "What's the latest technology news from March 2024?"
response = self_query_rag(query)
print(response)

I'm sorry, but without any relevant documents matching the criteria, I am unable to provide the latest technology news from March 2024. I recommend checking reputable technology news websites or sources for the most up-to-date information.


8. Adaptive RAG
Adjusts retrieval and augmentation based on the query complexity.

In [8]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from enum import Enum

# Initialize components
embeddings = OpenAIEmbeddings()
llm = ChatOpenAI(model_name="gpt-3.5-turbo")

# Sample knowledge base
documents = [
    "JavaScript is a high-level programming language primarily used for web development.",
    "React is a JavaScript library for building user interfaces, especially single-page applications.",
    "Node.js enables JavaScript to run on the server side, outside the browser environment.",
    "Express.js is a web application framework for Node.js designed for building web applications and APIs.",
    "Redux is a predictable state container for JavaScript apps, often used with React.",
    "Angular is a TypeScript-based open-source web application framework led by the Angular Team at Google.",
    "Vue.js is a progressive JavaScript framework for building user interfaces and single-page applications.",
    "TypeScript adds static type definitions to JavaScript, enhancing code quality and understandability.",
    "Webpack is a static module bundler for modern JavaScript applications.",
    "Babel is a JavaScript compiler that allows developers to use next-generation JavaScript features."
]
vectorstore = FAISS.from_texts(documents, embeddings)

# Query complexity assessment prompt
complexity_template = """Analyze this query and determine its complexity and the best retrieval strategy.

Query: {query}

Return a JSON with:
- "complexity": "simple", "medium", or "complex"
- "requires_retrieval": true or false (if the query needs external knowledge)
- "k_documents": number of documents to retrieve (1-5)"""
complexity_prompt = ChatPromptTemplate.from_template(complexity_template)

# Different RAG prompts based on complexity
simple_template = """Answer the question directly:
Question: {question}
Answer:"""
simple_prompt = ChatPromptTemplate.from_template(simple_template)

standard_template = """Answer the question based on this context:
Context: {context}
Question: {question}
Answer:"""
standard_prompt = ChatPromptTemplate.from_template(standard_template)

complex_template = """Answer this complex question step by step using the provided context:
Context: {context}
Question: {question}

Think through this carefully:
1. What are the key concepts in the question?
2. What information from the context is relevant?
3. What additional reasoning is needed?

Now provide a detailed answer:"""
complex_prompt = ChatPromptTemplate.from_template(complex_template)

# Adaptive RAG function
def adaptive_rag(query):
    # Step 1: Assess query complexity
    complexity_chain = complexity_prompt | llm | StrOutputParser()
    complexity_assessment = complexity_chain.invoke({"query": query})
    
    import json
    import re
    
    # Extract JSON from potential text output
    json_match = re.search(r'{.*}', complexity_assessment, re.DOTALL)
    if json_match:
        try:
            assessment = json.loads(json_match.group())
        except:
            # Default values if JSON parsing fails
            assessment = {
                "complexity": "medium",
                "requires_retrieval": True,
                "k_documents": 3
            }
    else:
        assessment = {
            "complexity": "medium",
            "requires_retrieval": True,
            "k_documents": 3
        }
    
    # Step 2: Determine retrieval strategy
    if not assessment.get("requires_retrieval", True):
        # Direct response without retrieval
        response_chain = simple_prompt | llm | StrOutputParser()
        response = response_chain.invoke({"question": query})
        return response
    
    # Step 3: Retrieve with adaptive parameters
    k_docs = min(max(assessment.get("k_documents", 3), 1), 5)  # Ensure between 1-5
    retriever = vectorstore.as_retriever(search_kwargs={"k": k_docs})
    docs = retriever.get_relevant_documents(query)
    context = "\n".join([doc.page_content for doc in docs])
    
    # Step 4: Select appropriate prompt based on complexity
    complexity_level = assessment.get("complexity", "medium").lower()
    
    if complexity_level == "simple":
        prompt = standard_prompt
    elif complexity_level == "complex":
        prompt = complex_prompt
    else:  # medium
        prompt = standard_prompt
    
    # Step 5: Generate response with selected prompt
    response_chain = prompt | llm | StrOutputParser()
    response = response_chain.invoke({"context": context, "question": query})
    
    return response

# Example queries of varying complexity
simple_query = "What is JavaScript?"
medium_query = "How does Node.js relate to JavaScript?"
complex_query = "Compare and contrast React, Angular, and Vue.js approaches to state management."

# Try with a query
response = adaptive_rag(medium_query)
print(response)

Node.js allows JavaScript to be used on the server side, expanding its capabilities beyond just the browser environment.
